<a href="https://colab.research.google.com/github/zgander/flyrank_ml/blob/main/work/notebooks/w03_data_contract.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/zgander/flyrank_ml/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [1]:
# ============================================================
# W03 — DuckDB + Hugging Face Warehouse Setup
# ============================================================

!pip -q install duckdb

import duckdb
from google.colab import userdata

# ------------------------------------------------------------
# 1. Create DuckDB connection
# ------------------------------------------------------------

con = duckdb.connect()

# ------------------------------------------------------------
# 2. Get Hugging Face token from Colab Secrets
# ------------------------------------------------------------

hf_token = userdata.get("HF_TOKEN")

if not hf_token:
    raise ValueError(
        "HF_TOKEN was not found.\n\n"
        "Go to Colab → Secrets → Add a new secret:\n"
        "Name: HF_TOKEN\n"
        "Value: your Hugging Face READ token"
    )

# ------------------------------------------------------------
# 3. Store token securely in DuckDB
# ------------------------------------------------------------

con.execute(
    "SET VARIABLE hf_token = ?",
    [hf_token]
)

con.execute("""
    CREATE OR REPLACE SECRET hf (
        TYPE huggingface,
        TOKEN getvariable('hf_token')
    )
""")

# ------------------------------------------------------------
# 4. Define warehouse paths
# ------------------------------------------------------------

REL = "hf://datasets/FlyRank/internship-warehouse"

FACT = f"{REL}/fact_content_daily_performance"

DIM_CONTENT = f"{REL}/dim_content.parquet"
DIM_CLIENTS = f"{REL}/dim_clients.parquet"

# Feature window: February 2026
FEB = f"{FACT}/month=2026-02/*.parquet"

# Label window: March 2026
MAR = f"{FACT}/month=2026-03/*.parquet"

print("✓ DuckDB connected")
print("✓ Hugging Face authentication configured")
print("✓ Warehouse paths configured")
print()
print("Feature window:", "February 2026")
print("Label window:", "March 2026")

✓ DuckDB connected
✓ Hugging Face authentication configured
✓ Warehouse paths configured

Feature window: February 2026
Label window: March 2026


In [2]:
# Search W03's executed-cell history for DuckDB / warehouse code

history = get_ipython().history_manager.input_hist_raw

print("=== W03 execution history containing DuckDB / warehouse references ===\n")

for i, code in enumerate(history):
    code_lower = code.lower()

    if any(term in code_lower for term in [
        "duckdb",
        "read_parquet",
        "fact_content_daily_performance",
        "hf_token",
        "huggingface",
        "content_daily",
        "dim_clients"
    ]):
        print(f"\n{'='*80}")
        print(f"CELL {i}")
        print(f"{'='*80}")
        print(code)

=== W03 execution history containing DuckDB / warehouse references ===


CELL 1
# ============================================================
# W03 — DuckDB + Hugging Face Warehouse Setup
# ============================================================

!pip -q install duckdb

import duckdb
from google.colab import userdata

# ------------------------------------------------------------
# 1. Create DuckDB connection
# ------------------------------------------------------------

con = duckdb.connect()

# ------------------------------------------------------------
# 2. Get Hugging Face token from Colab Secrets
# ------------------------------------------------------------

hf_token = userdata.get("HF_TOKEN")

if not hf_token:
    raise ValueError(
        "HF_TOKEN was not found.\n\n"
        "Go to Colab → Secrets → Add a new secret:\n"
        "Name: HF_TOKEN\n"
        "Value: your Hugging Face READ token"
    )

# ------------------------------------------------------------
# 3. Sto

## 1. Unit of analysis + time window

**Unit of analysis:** One row represents one content item for one client (`client_hash_id` × `content_hash_id`).

The source table, `fact_content_daily_performance`, is at daily grain: one row per client × content × day. For this project, I aggregate those daily observations into one row per client × content for a defined decision point.

**Feature window:** February 2026 (`2026-02-01` → `2026-02-28`). These are the signals that would have been knowable at the decision point.

**Label window:** March 2026 (`2026-03-01` → `2026-03-31`). This is the subsequent outcome window used to evaluate the page after the decision point.

The feature and label windows do not overlap. June 2026 is kept as a sealed final month rather than being used while developing the label logic.


In [3]:
grain_check = con.sql(f"""
    SELECT
        report_date,
        client_hash_id,
        content_hash_id,
        COUNT(*) AS n
    FROM read_parquet('{FACT}/month=2026-02/*.parquet')
    GROUP BY
        report_date,
        client_hash_id,
        content_hash_id
    HAVING COUNT(*) > 1
    LIMIT 5
""").df()

print("Duplicate client × content × day keys:")
display(grain_check)

assert grain_check.empty, "Duplicate grain keys found — investigate before aggregating."

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Duplicate client × content × day keys:


,report_date,client_hash_id,content_hash_id,n


## 2. Fields: feature / label / context / excluded

### Feature

The features are signals that would be available at the decision point, using the February 2026 feature window:

- `gsc_impressions` — total Google Search Console impressions during February.
- `gsc_clicks` — total GSC clicks during February.
- `gsc_ctr` — aggregate click-through rate during February, calculated as clicks / impressions.
- `gsc_avg_position` — average search position during February.
- `content_age_days` — age of the content at the decision point.

These features describe the page's observed search visibility, traffic, and age before the label window.

### Label

The label is:

- `went_dark` — whether the content item received zero measured GSC clicks during March 2026, among content items for which GSC data was available during the label window.

March is strictly after the February feature window, so the label represents a subsequent observed outcome rather than an input available at prediction time.

### Context

The following fields are retained for grouping, joining, validation, and interpretation but are not model features:

- `client_hash_id` — identifies the pseudonymized client.
- `content_hash_id` — identifies the pseudonymized content item.
- `report_date` — identifies the daily observation and is used to construct the feature and label windows.

The identifiers are used for grouping and joins only, not as predictive features.

### Excluded

The following fields are excluded from the model:

- `trend_direction` — excluded because it is derived from `trend_pct` and therefore would duplicate information used to define a decline-related label.
- `trend_pct` — excluded because it is a derived trend signal and can introduce leakage depending on how its window is constructed.
- March GSC/GA4 metrics — excluded because they occur after the February decision point and therefore would expose future information.
- Product-derived scores such as `health_score` or `priority_score` — excluded because they represent pre-existing/product-generated decisions rather than independent observable signals.
- `action_type` — excluded because it represents an action/decision rather than an input available for predicting the outcome.

In [4]:
# Verify that the fields we plan to use actually exist
# and inspect their data types.

columns_check = con.sql(f"""
    DESCRIBE SELECT *
    FROM read_parquet('{FEB}')
""").df()

display(
    columns_check[
        columns_check["column_name"].isin([
            "client_hash_id",
            "content_hash_id",
            "report_date",
            "gsc_impressions",
            "gsc_clicks",
            "gsc_avg_position",
            "ga4_sessions",
            "gsc_data_available",
            "ga4_data_available"
        ])
    ]
)

# Confirm that the key fields needed for the contract are present.

required_fields = {
    "client_hash_id",
    "content_hash_id",
    "report_date",
    "gsc_impressions",
    "gsc_clicks",
    "gsc_avg_position",
    "gsc_data_available"
}

available_fields = set(columns_check["column_name"])

missing_fields = required_fields - available_fields

print("Required fields:", required_fields)
print("Missing fields:", missing_fields)

assert not missing_fields, f"Missing required fields: {missing_fields}"

print("✓ All required contract fields are present.")

,column_name,column_type,null,key,default,extra
0,report_date,DATE,YES,None,None,None
1,client_hash_id,VARCHAR,YES,None,None,None
2,content_hash_id,VARCHAR,YES,None,None,None
5,gsc_data_available,BOOLEAN,YES,None,None,None
6,ga4_data_available,BOOLEAN,YES,None,None,None
7,gsc_impressions,BIGINT,YES,None,None,None
8,gsc_clicks,BIGINT,YES,None,None,None
10,gsc_avg_position,DOUBLE,YES,None,None,None
12,ga4_sessions,BIGINT,YES,None,None,None


Required fields: {'gsc_clicks', 'client_hash_id', 'gsc_impressions', 'report_date', 'gsc_data_available', 'content_hash_id', 'gsc_avg_position'}
Missing fields: set()
✓ All required contract fields are present.


## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

In [5]:
# Section 3 — Verify the data contract with DuckDB queries

print("=" * 70)
print("1. GRAIN CHECK — client × content × day")
print("=" * 70)

grain_check = con.sql(f"""
    SELECT
        report_date,
        client_hash_id,
        content_hash_id,
        COUNT(*) AS n
    FROM read_parquet('{FEB}')
    GROUP BY
        report_date,
        client_hash_id,
        content_hash_id
    HAVING COUNT(*) > 1
    LIMIT 5
""").df()

display(grain_check)

assert grain_check.empty, "Duplicate client × content × day rows found."
print("✓ No duplicate client × content × day keys found.")


print("\n" + "=" * 70)
print("2. FEATURE WINDOW — February 2026")
print("=" * 70)

feb_window = con.sql(f"""
    SELECT
        COUNT(*) AS rows,
        COUNT(DISTINCT client_hash_id) AS clients,
        COUNT(DISTINCT content_hash_id) AS content_items,
        MIN(report_date) AS min_date,
        MAX(report_date) AS max_date
    FROM read_parquet('{FEB}')
""").df()

display(feb_window)


print("\n" + "=" * 70)
print("3. LABEL WINDOW — March 2026")
print("=" * 70)

mar_window = con.sql(f"""
    SELECT
        COUNT(*) AS rows,
        COUNT(DISTINCT client_hash_id) AS clients,
        COUNT(DISTINCT content_hash_id) AS content_items,
        MIN(report_date) AS min_date,
        MAX(report_date) AS max_date
    FROM read_parquet('{MAR}')
""").df()

display(mar_window)


print("\n" + "=" * 70)
print("4. GSC DATA AVAILABILITY — February")
print("=" * 70)

feb_availability = con.sql(f"""
    SELECT
        COUNT(*) AS total_rows,
        COUNT(*) FILTER (
            WHERE gsc_data_available IS TRUE
        ) AS gsc_available_rows,
        COUNT(*) FILTER (
            WHERE gsc_data_available IS FALSE
        ) AS gsc_unavailable_rows
    FROM read_parquet('{FEB}')
""").df()

display(feb_availability)


print("\n" + "=" * 70)
print("5. GSC DATA AVAILABILITY — March")
print("=" * 70)

mar_availability = con.sql(f"""
    SELECT
        COUNT(*) AS total_rows,
        COUNT(*) FILTER (
            WHERE gsc_data_available IS TRUE
        ) AS gsc_available_rows,
        COUNT(*) FILTER (
            WHERE gsc_data_available IS FALSE
        ) AS gsc_unavailable_rows
    FROM read_parquet('{MAR}')
""").df()

display(mar_availability)


print("\n" + "=" * 70)
print("6. MISSING VALUES — February feature fields")
print("=" * 70)

missing_check = con.sql(f"""
    SELECT
        COUNT(*) AS total_rows,

        COUNT(*) FILTER (
            WHERE gsc_impressions IS NULL
        ) AS missing_gsc_impressions,

        COUNT(*) FILTER (
            WHERE gsc_clicks IS NULL
        ) AS missing_gsc_clicks,

        COUNT(*) FILTER (
            WHERE gsc_avg_position IS NULL
        ) AS missing_gsc_avg_position,

        COUNT(*) FILTER (
            WHERE gsc_data_available IS NULL
        ) AS missing_gsc_availability_flag

    FROM read_parquet('{FEB}')
""").df()

display(missing_check)


print("\n" + "=" * 70)
print("7. FEATURE/LABEL WINDOW OVERLAP CHECK")
print("=" * 70)

overlap_check = con.sql(f"""
    SELECT
        COUNT(*) AS overlapping_dates
    FROM (
        SELECT DISTINCT report_date
        FROM read_parquet('{FEB}')
    ) feb
    INNER JOIN (
        SELECT DISTINCT report_date
        FROM read_parquet('{MAR}')
    ) mar
    ON feb.report_date = mar.report_date
""").df()

display(overlap_check)

assert overlap_check.iloc[0]["overlapping_dates"] == 0, \
    "Feature and label windows overlap."

print("✓ February and March windows do not overlap.")


print("\n" + "=" * 70)
print("8. FINAL WINDOW SUMMARY")
print("=" * 70)

print("Feature window : 2026-02-01 → 2026-02-28")
print("Label window   : 2026-03-01 → 2026-03-31")
print("Feature grain  : client × content")
print("Raw fact grain : client × content × day")
print("✓ Section 3 verification complete.")

1. GRAIN CHECK — client × content × day


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,report_date,client_hash_id,content_hash_id,n


✓ No duplicate client × content × day keys found.

2. FEATURE WINDOW — February 2026


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

...

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,total_rows,missing_gsc_impressions,missing_gsc_clicks,missing_gsc_avg_position,missing_gsc_availability_flag
0,7355108,92256,92256,4733326,92256



7. FEATURE/LABEL WINDOW OVERLAP CHECK


,overlapping_dates
0,0


✓ February and March windows do not overlap.

8. FINAL WINDOW SUMMARY
Feature window : 2026-02-01 → 2026-02-28
Label window   : 2026-03-01 → 2026-03-31
Feature grain  : client × content
Raw fact grain : client × content × day
✓ Section 3 verification complete.


## 4. Data limits

### Data limits

This dataset cannot tell us whether refreshing a page will *cause* its performance to improve. Our label is an observed future outcome, not a causal measurement of the effect of a refresh.

The warehouse is an **unbalanced panel**: different clients have different amounts of historical data. Therefore, not every client or content item necessarily has sufficient observations in both the feature and label windows. The resulting dataset may therefore represent only the subset with adequate measured history.

Some rows contain **GSC data without GA4 data** because GA4 tracking starts at different times for different clients. A GA4 value of zero before `ga4_data_start` must not be interpreted as zero engagement. The `ga4_data_available` flag must be used when GA4 features are included.

The February feature window and March label window must remain strictly separated. Any feature derived from March or later would expose future information and create data leakage.

The data also cannot establish why a page's performance changed. We observe search and engagement outcomes, but we do not observe every external factor that may have influenced them.

Finally, the model can identify pages that resemble pages with a particular historical outcome, but it cannot guarantee that the recommended action (refresh, expand, protect, prune, or monitor) will produce that outcome.

In [6]:
# Section 4 — Verify important data limitations

print("=" * 70)
print("1. CLIENT HISTORY COVERAGE")
print("=" * 70)

client_history = con.sql(f"""
    SELECT
        COUNT(*) AS clients,
        COUNT(*) FILTER (
            WHERE gsc_data_start <= DATE '2026-02-01'
        ) AS clients_with_gsc_before_feature_window,
        COUNT(*) FILTER (
            WHERE gsc_data_start > DATE '2026-02-01'
        ) AS clients_starting_after_feature_window
    FROM read_parquet('{DIM_CLIENTS}')
""").df()

display(client_history)


print("=" * 70)
print("2. GA4 AVAILABILITY IN FEBRUARY")
print("=" * 70)

ga4_availability = con.sql(f"""
    SELECT
        COUNT(*) AS total_rows,
        COUNT(*) FILTER (
            WHERE ga4_data_available IS TRUE
        ) AS ga4_available_rows,
        COUNT(*) FILTER (
            WHERE ga4_data_available IS FALSE
        ) AS ga4_unavailable_rows
    FROM read_parquet('{FEB}')
""").df()

display(ga4_availability)


print("=" * 70)
print("3. FEATURE / LABEL WINDOW SEPARATION")
print("=" * 70)

windows = con.sql(f"""
    SELECT
        MIN(report_date) AS feature_start,
        MAX(report_date) AS feature_end
    FROM read_parquet('{FEB}')
""").df()

label_windows = con.sql(f"""
    SELECT
        MIN(report_date) AS label_start,
        MAX(report_date) AS label_end
    FROM read_parquet('{MAR}')
""").df()

display(windows)
display(label_windows)

feature_end = windows.iloc[0]["feature_end"]
label_start = label_windows.iloc[0]["label_start"]

assert feature_end < label_start, (
    "Feature and label windows overlap — potential leakage."
)

print("✓ Feature window ends before label window begins.")


1. CLIENT HISTORY COVERAGE


,clients,clients_with_gsc_before_feature_window,clients_starting_after_feature_window
0,104,41,26


2. GA4 AVAILABILITY IN FEBRUARY


,total_rows,ga4_available_rows,ga4_unavailable_rows
0,7355108,145321,3057464


3. FEATURE / LABEL WINDOW SEPARATION


,feature_start,feature_end
0,2026-02-01,2026-02-28


,label_start,label_end
0,2026-03-01,2026-03-31


✓ Feature window ends before label window begins.


In [7]:
# ============================================================
# W03 → W05 MODELING DATASET
# Decision grain: one row per client × content
# Feature window: February 2026
# Label window: March 2026
# ============================================================

model_data = con.sql(f"""
WITH february AS (
    SELECT
        client_hash_id,
        content_hash_id,

        -- ----------------------------------------------------
        -- February GSC features
        -- ----------------------------------------------------

        SUM(
            CASE
                WHEN gsc_data_available IS TRUE
                THEN COALESCE(gsc_impressions, 0)
                ELSE 0
            END
        ) AS feb_impressions,

        SUM(
            CASE
                WHEN gsc_data_available IS TRUE
                THEN COALESCE(gsc_clicks, 0)
                ELSE 0
            END
        ) AS feb_clicks,

        -- CTR as percentage
        -- Example: 100 clicks / 10,000 impressions = 1.0%
        100.0 *
        SUM(
            CASE
                WHEN gsc_data_available IS TRUE
                THEN COALESCE(gsc_clicks, 0)
                ELSE 0
            END
        )
        /
        NULLIF(
            SUM(
                CASE
                    WHEN gsc_data_available IS TRUE
                    THEN COALESCE(gsc_impressions, 0)
                    ELSE 0
                END
            ),
            0
        ) AS feb_ctr,

        -- Impression-weighted average position.
        -- Position = 0 means no position data, so exclude it.
        SUM(
            CASE
                WHEN gsc_data_available IS TRUE
                     AND COALESCE(gsc_avg_position, 0) > 0
                THEN gsc_avg_position * COALESCE(gsc_impressions, 0)
                ELSE 0
            END
        )
        /
        NULLIF(
            SUM(
                CASE
                    WHEN gsc_data_available IS TRUE
                         AND COALESCE(gsc_avg_position, 0) > 0
                    THEN COALESCE(gsc_impressions, 0)
                    ELSE 0
                END
            ),
            0
        ) AS feb_avg_position,

        -- Number of days with measured GSC data
        COUNT(*) FILTER (
            WHERE gsc_data_available IS TRUE
        ) AS feb_gsc_available_days,

        -- ----------------------------------------------------
        -- February GA4 features
        -- ----------------------------------------------------

        SUM(
            CASE
                WHEN ga4_data_available IS TRUE
                THEN COALESCE(ga4_sessions, 0)
                ELSE 0
            END
        ) AS feb_ga4_sessions,

        COUNT(*) FILTER (
            WHERE ga4_data_available IS TRUE
        ) AS feb_ga4_available_days

    FROM read_parquet('{FEB}')

    GROUP BY
        client_hash_id,
        content_hash_id
),

march AS (
    SELECT
        client_hash_id,
        content_hash_id,

        -- March clicks are used only to construct the label.
        SUM(
            CASE
                WHEN gsc_data_available IS TRUE
                THEN COALESCE(gsc_clicks, 0)
                ELSE 0
            END
        ) AS mar_clicks,

        -- Number of days with measured GSC data
        COUNT(*) FILTER (
            WHERE gsc_data_available IS TRUE
        ) AS mar_gsc_available_days

    FROM read_parquet('{MAR}')

    GROUP BY
        client_hash_id,
        content_hash_id
)

SELECT
    f.client_hash_id,
    f.content_hash_id,

    -- February features
    f.feb_impressions,
    f.feb_clicks,
    f.feb_ctr,
    f.feb_avg_position,
    f.feb_gsc_available_days,
    f.feb_ga4_sessions,
    f.feb_ga4_available_days,

    -- March outcome-audit fields
    m.mar_clicks,
    m.mar_gsc_available_days,

    -- --------------------------------------------------------
    -- Target
    --
    -- 1 = measured GSC data existed in March,
    --     but the content received zero clicks.
    --
    -- 0 = the content received at least one measured click.
    -- --------------------------------------------------------
    CASE
        WHEN m.mar_gsc_available_days > 0
             AND COALESCE(m.mar_clicks, 0) = 0
        THEN 1
        ELSE 0
    END AS went_dark

FROM february f

INNER JOIN march m
    ON f.client_hash_id = m.client_hash_id
    AND f.content_hash_id = m.content_hash_id

-- Require measured GSC data during both windows.
WHERE f.feb_gsc_available_days > 0
  AND m.mar_gsc_available_days > 0
""").df()

print("✓ Modeling dataset created.")
print(f"Shape: {model_data.shape}")

display(model_data.head())

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

✓ Modeling dataset created.
Shape: (134238, 12)


,client_hash_id,content_hash_id,feb_impressions,feb_clicks,feb_ctr,feb_avg_position,feb_gsc_available_days,feb_ga4_sessions,feb_ga4_available_days,mar_clicks,mar_gsc_available_days,went_dark
0,client_e547b89c05043229,content_7995404695ee1ffd,1012.0,3.0,0.296443,28.886364,28,5.0,4,1.0,29,0
1,client_e547b89c05043229,content_ccbb253f142217c3,1598.0,7.0,0.438048,17.273467,28,13.0,9,6.0,29,0
2,client_e547b89c05043229,content_ae16a6b9cf64c80a,861.0,0.0,0.000000,8.182346,28,1.0,1,0.0,29,1
3,client_e547b89c05043229,content_acf700633f016e5a,245.0,0.0,0.000000,8.410256,28,1.0,1,0.0,29,1
4,client_e547b89c05043229,content_712e44562fed8ff2,306.0,0.0,0.000000,8.000000,28,0.0,0,1.0,29,0


In [8]:
# ============================================================
# MODELING DATASET — LABEL & GRAIN VALIDATION
# ============================================================

import pandas as pd

print("=" * 70)
print("1. DATASET OVERVIEW")
print("=" * 70)

print(f"Rows:            {len(model_data):,}")
print(f"Columns:         {len(model_data.columns)}")
print(f"Clients:         {model_data['client_hash_id'].nunique():,}")
print(f"Content items:   {model_data['content_hash_id'].nunique():,}")


print("\n" + "=" * 70)
print("2. LABEL DISTRIBUTION")
print("=" * 70)

label_counts = model_data["went_dark"].value_counts().sort_index()

display(
    pd.DataFrame({
        "count": label_counts,
        "percentage": label_counts / len(model_data) * 100
    })
)


print("\n" + "=" * 70)
print("3. DUPLICATE GRAIN CHECK")
print("=" * 70)

duplicates = model_data.duplicated(
    subset=["client_hash_id", "content_hash_id"]
).sum()

print(f"Duplicate client × content rows: {duplicates:,}")

assert duplicates == 0, \
    "Duplicate client × content rows found."

print("✓ One row per client × content.")


print("\n" + "=" * 70)
print("4. MISSING VALUES")
print("=" * 70)

missing = (
    model_data
    .isna()
    .sum()
    .sort_values(ascending=False)
)

display(
    missing.to_frame("missing_rows")
)


print("\n" + "=" * 70)
print("5. LABEL SANITY CHECK")
print("=" * 70)

label_summary = model_data.groupby("went_dark").agg(
    rows=("went_dark", "size"),
    median_feb_impressions=("feb_impressions", "median"),
    median_feb_clicks=("feb_clicks", "median"),
    median_feb_ctr=("feb_ctr", "median"),
    median_feb_position=("feb_avg_position", "median"),
    median_mar_clicks=("mar_clicks", "median"),
    median_mar_gsc_days=("mar_gsc_available_days", "median")
)

display(label_summary)


print("\n" + "=" * 70)
print("6. ZERO-EXPOSURE CHECK")
print("=" * 70)

zero_impressions = (
    model_data["feb_impressions"] == 0
).sum()

zero_clicks = (
    model_data["feb_clicks"] == 0
).sum()

print(f"Rows with 0 February impressions: {zero_impressions:,}")
print(f"Rows with 0 February clicks:      {zero_clicks:,}")

print("\n✓ Validation complete.")

1. DATASET OVERVIEW
Rows:            134,238
Columns:         12
Clients:         42
Content items:   134,238

2. LABEL DISTRIBUTION


,count,percentage
went_dark,,
0,57943,43.16438
1,76295,56.83562



3. DUPLICATE GRAIN CHECK
Duplicate client × content rows: 0
✓ One row per client × content.

4. MISSING VALUES


,missing_rows
feb_avg_position,932
client_hash_id,0
feb_impressions,0
content_hash_id,0
feb_clicks,0
feb_ctr,0
feb_gsc_available_days,0
feb_ga4_sessions,0
feb_ga4_available_days,0
mar_clicks,0



5. LABEL SANITY CHECK


,rows,median_feb_impressions,median_feb_clicks,median_feb_ctr,median_feb_position,median_mar_clicks,median_mar_gsc_days
went_dark,,,,,,,
0,57943,1081.0,2.0,0.201991,6.447553,3.0,31.0
1,76295,41.0,0.0,0.000000,9.714286,0.0,22.0



6. ZERO-EXPOSURE CHECK
Rows with 0 February impressions: 0
Rows with 0 February clicks:      80,872

✓ Validation complete.


In [9]:
# ============================================================
# LABEL DIAGNOSTIC — WENT DARK BY FEBRUARY DEMAND
# ============================================================

model_data["impression_bucket"] = pd.cut(
    model_data["feb_impressions"],
    bins=[-1, 10, 50, 100, 500, 1000, 5000, float("inf")],
    labels=[
        "0–10",
        "11–50",
        "51–100",
        "101–500",
        "501–1K",
        "1K–5K",
        "5K+"
    ]
)

demand_label_check = (
    model_data
    .groupby("impression_bucket", observed=False)
    .agg(
        rows=("went_dark", "size"),
        went_dark_count=("went_dark", "sum"),
        went_dark_rate=("went_dark", "mean"),
        median_feb_clicks=("feb_clicks", "median"),
        median_feb_ctr=("feb_ctr", "median")
    )
)

demand_label_check["went_dark_rate"] *= 100

display(demand_label_check)

# ============================================================
# LABEL DIAGNOSTIC — HIGH-DEMAND PAGES
# ============================================================

high_demand = model_data[
    model_data["feb_impressions"] >= 100
].copy()

print(f"Pages with >=100 February impressions: {len(high_demand):,}")
print(
    f"Went-dark rate among high-demand pages: "
    f"{high_demand['went_dark'].mean() * 100:.2f}%"
)

high_demand_500 = model_data[
    model_data["feb_impressions"] >= 500
].copy()

print(f"\nPages with >=500 February impressions: {len(high_demand_500):,}")
print(
    f"Went-dark rate among >=500-impression pages: "
    f"{high_demand_500['went_dark'].mean() * 100:.2f}%"
)

,rows,went_dark_count,went_dark_rate,median_feb_clicks,median_feb_ctr
impression_bucket,,,,,
0–10,23967,22167,92.489673,0.0,0.000000
11–50,21847,18754,85.842450,0.0,0.000000
51–100,11901,9425,79.195026,0.0,0.000000
101–500,30398,18809,61.875781,0.0,0.000000
501–1K,13236,4403,33.265337,1.0,0.154083
1K–5K,24563,2640,10.747873,4.0,0.213106
5K+,8326,97,1.165025,23.0,0.241398


Pages with >=100 February impressions: 76,702
Went-dark rate among high-demand pages: 34.01%

Pages with >=500 February impressions: 46,152
Went-dark rate among >=500-impression pages: 15.50%


In [10]:
# ============================================================
# FINAL W05 MODELING POPULATION
# ============================================================

W05_MIN_IMPRESSIONS = 500

w05_data = model_data[
    model_data["feb_impressions"] >= W05_MIN_IMPRESSIONS
].copy()

# Remove the diagnostic bucket — it is not a model feature.
if "impression_bucket" in w05_data.columns:
    w05_data = w05_data.drop(columns=["impression_bucket"])

print("=" * 70)
print("FINAL W05 MODELING DATASET")
print("=" * 70)

print(f"Minimum February impressions: {W05_MIN_IMPRESSIONS:,}")
print(f"Rows:                         {len(w05_data):,}")
print(f"Clients:                      {w05_data['client_hash_id'].nunique():,}")
print(f"Went-dark rows:               {w05_data['went_dark'].sum():,}")
print(f"Went-dark rate:               {w05_data['went_dark'].mean() * 100:.2f}%")

print("\nLabel distribution:")
display(
    w05_data["went_dark"]
    .value_counts()
    .sort_index()
    .to_frame("count")
    .assign(
        percentage=lambda x: x["count"] / len(w05_data) * 100
    )
)

print("\n✓ W05 population created.")

FINAL W05 MODELING DATASET
Minimum February impressions: 500
Rows:                         46,152
Clients:                      31
Went-dark rows:               7,153
Went-dark rate:               15.50%

Label distribution:


,count,percentage
went_dark,,
0,38999,84.501213
1,7153,15.498787



✓ W05 population created.


In [11]:
# ============================================================
# FINAL W05 DATASET VALIDATION
# ============================================================

print("=" * 70)
print("W05 DATASET VALIDATION")
print("=" * 70)

# Grain
duplicates = w05_data.duplicated(
    subset=["client_hash_id", "content_hash_id"]
).sum()

print(f"Duplicate client × content rows: {duplicates}")

assert duplicates == 0

# Eligibility
below_threshold = (
    w05_data["feb_impressions"] < W05_MIN_IMPRESSIONS
).sum()

print(f"Rows below impression threshold: {below_threshold}")

assert below_threshold == 0

# Availability
print(
    f"Rows with February GSC data: "
    f"{(w05_data['feb_gsc_available_days'] > 0).sum():,}"
)

print(
    f"Rows with March GSC data: "
    f"{(w05_data['mar_gsc_available_days'] > 0).sum():,}"
)

# Target leakage check
print("\nColumns:")
print(list(w05_data.columns))

print("\n✓ Final W05 dataset passes basic validation.")

W05 DATASET VALIDATION
Duplicate client × content rows: 0
Rows below impression threshold: 0
Rows with February GSC data: 46,152
Rows with March GSC data: 46,152

Columns:
['client_hash_id', 'content_hash_id', 'feb_impressions', 'feb_clicks', 'feb_ctr', 'feb_avg_position', 'feb_gsc_available_days', 'feb_ga4_sessions', 'feb_ga4_available_days', 'mar_clicks', 'mar_gsc_available_days', 'went_dark']

✓ Final W05 dataset passes basic validation.


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.